In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

import xgboost as xgb

import joblib
import warnings
warnings.filterwarnings('ignore')


In [46]:
file_path = r"C:\Users\moham\Downloads\arabic_gestures_corrected.csv"

df = pd.read_csv(file_path)

print("="*50)
print("DATASET OVERVIEW")
print("="*50)
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nDataset Info:")
print(df.info())
print(f"\nMissing Values:\n{df.isnull().sum().sum()} total missing values")

DATASET OVERVIEW
Shape: (4800, 23)

First 5 rows:
   left_flex_1  left_flex_2  left_flex_3  left_flex_4  left_flex_5  \
0          946          907          944          792          945   
1          944          909          945          793          943   
2          944          909          947          793          946   
3          944          908          944          795          944   
4          943          909          945          794          944   

   left_acc_x  left_acc_y  left_acc_z  left_gyro_x  left_gyro_y  ...  \
0       -2848       -3248      -15216          174          163  ...   
1       -2972       -3188      -15236            0           -3  ...   
2       -2956       -3320      -15308           36           47  ...   
3       -2932       -3204      -15176          204           76  ...   
4       -3124       -3344      -15312            5            6  ...   

   right_flex_3  right_flex_4  right_flex_5  right_acc_x  right_acc_y  \
0           745        

In [49]:
print("="*50)
print("LABEL ANALYSIS")
print("="*50)

print(f"Unique Labels: {df['label'].unique()}")
print(f"\nNumber of unique gestures: {df['label'].nunique()}")
print(f"\nLabel Distribution:")
label_counts = df['label'].value_counts()
print(label_counts)

plt.show()

LABEL ANALYSIS
Unique Labels: ['نتعرف' 'أنا' 'أسمي' 'سلام' 'عليكم' 'رحمة' 'الله' 'بركاته' 'أهلا' 'اتفضل'
 'سنك' 'عاوز' 'بحب' 'وعد' 'مهذب' 'صديق']

Number of unique gestures: 16

Label Distribution:
label
نتعرف     300
أنا       300
أسمي      300
سلام      300
عليكم     300
رحمة      300
الله      300
بركاته    300
أهلا      300
اتفضل     300
سنك       300
عاوز      300
بحب       300
وعد       300
مهذب      300
صديق      300
Name: count, dtype: int64


In [ ]:
feature_columns = [
    'left_flex_1', 'left_flex_2', 'left_flex_3', 'left_flex_4', 'left_flex_5',
    'left_acc_x', 'left_acc_y', 'left_acc_z',
    'left_gyro_x', 'left_gyro_y', 'left_gyro_z',
    
    'right_flex_1', 'right_flex_2', 'right_flex_3', 'right_flex_4', 'right_flex_5',
    'right_acc_x', 'right_acc_y', 'right_acc_z',
    'right_gyro_x', 'right_gyro_y', 'right_gyro_z'
]

X = df[feature_columns]
y = df['label']

print(f"Features shape: {X.shape}")
print(f"Number of features: {len(feature_columns)}")
print(f"\nFeature names:")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nLabels shape: {y.shape}")
print(f"Unique labels: {y.nunique()}")

Features shape: (4800, 22)
Number of features: 22

Feature names:
   1. left_flex_1
   2. left_flex_2
   3. left_flex_3
   4. left_flex_4
   5. left_flex_5
   6. left_acc_x
   7. left_acc_y
   8. left_acc_z
   9. left_gyro_x
  10. left_gyro_y
  11. left_gyro_z
  12. right_flex_1
  13. right_flex_2
  14. right_flex_3
  15. right_flex_4
  16. right_flex_5
  17. right_acc_x
  18. right_acc_y
  19. right_acc_z
  20. right_gyro_x
  21. right_gyro_y
  22. right_gyro_z

Labels shape: (4800,)
Unique labels: 16


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Features scaled successfully")
print(f"X_scaled shape: {X_scaled.shape}")
print(f"X_scaled mean: {X_scaled.mean():.6f}")
print(f"X_scaled std: {X_scaled.std():.6f}")

print(f"\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

Features scaled successfully
X_scaled shape: (4800, 22)
X_scaled mean: -0.000000
X_scaled std: 1.000000

Label encoding mapping:
  أسمي -> 0
  أنا -> 1
  أهلا -> 2
  اتفضل -> 3
  الله -> 4
  بحب -> 5
  بركاته -> 6
  رحمة -> 7
  سلام -> 8
  سنك -> 9
  صديق -> 10
  عاوز -> 11
  عليكم -> 12
  مهذب -> 13
  نتعرف -> 14
  وعد -> 15


In [ ]:
frames_per_session = 25  

df['session_id'] = np.arange(len(df)) // frames_per_session

print(f"Number of sessions: {df['session_id'].nunique()}")
print(f"Frames per session: {frames_per_session}")
print(f"\nFirst few sessions:")
print(df[['label', 'session_id']].head(20))

# Check session distribution
session_counts = df.groupby('session_id')['label'].nunique()
print(f"\nSessions with single label: {(session_counts == 1).sum()}/{len(session_counts)}")
print(f"Sessions with multiple labels: {(session_counts > 1).sum()}/{len(session_counts)}")

groups = df['session_id']

Number of sessions: 192
Frames per session: 25

First few sessions:
    label  session_id
0   نتعرف           0
1   نتعرف           0
2   نتعرف           0
3   نتعرف           0
4   نتعرف           0
5   نتعرف           0
6   نتعرف           0
7   نتعرف           0
8   نتعرف           0
9   نتعرف           0
10  نتعرف           0
11  نتعرف           0
12  نتعرف           0
13  نتعرف           0
14  نتعرف           0
15  نتعرف           0
16  نتعرف           0
17  نتعرف           0
18  نتعرف           0
19  نتعرف           0

Sessions with single label: 192/192
Sessions with multiple labels: 0/192


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X_scaled, y_encoded, groups))

X_train = X_scaled[train_idx]
X_test = X_scaled[test_idx]
y_train = y_encoded[train_idx]
y_test = y_encoded[test_idx]

print(f"Training set: {X_train.shape}")
print(f"Testing set: {X_test.shape}")
print(f"\nTraining labels distribution:")
train_dist = pd.Series(y_train).value_counts().sort_index()
for label_idx, count in train_dist.items():
    print(f"  {label_encoder.classes_[label_idx]}: {count} ({count/len(y_train)*100:.1f}%)")

print(f"\nTesting labels distribution:")
test_dist = pd.Series(y_test).value_counts().sort_index()
for label_idx, count in test_dist.items():
    print(f"  {label_encoder.classes_[label_idx]}: {count} ({count/len(y_test)*100:.1f}%)")

Training set: (3825, 22)
Testing set: (975, 22)

Training labels distribution:
  أسمي: 250 (6.5%)
  أنا: 200 (5.2%)
  أهلا: 275 (7.2%)
  اتفضل: 150 (3.9%)
  الله: 250 (6.5%)
  بحب: 225 (5.9%)
  بركاته: 275 (7.2%)
  رحمة: 150 (3.9%)
  سلام: 275 (7.2%)
  سنك: 300 (7.8%)
  صديق: 300 (7.8%)
  عاوز: 175 (4.6%)
  عليكم: 300 (7.8%)
  مهذب: 225 (5.9%)
  نتعرف: 250 (6.5%)
  وعد: 225 (5.9%)

Testing labels distribution:
  أسمي: 50 (5.1%)
  أنا: 100 (10.3%)
  أهلا: 25 (2.6%)
  اتفضل: 150 (15.4%)
  الله: 50 (5.1%)
  بحب: 75 (7.7%)
  بركاته: 25 (2.6%)
  رحمة: 150 (15.4%)
  سلام: 25 (2.6%)
  عاوز: 125 (12.8%)
  مهذب: 75 (7.7%)
  نتعرف: 50 (5.1%)
  وعد: 75 (7.7%)


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=200,           
    max_depth=6,                
    learning_rate=0.05,         
    min_child_weight=5,         
    
    subsample=0.8,             
    colsample_bytree=0.8,       
    colsample_bylevel=0.8,      
    reg_alpha=1.0,              
    reg_lambda=2.0,             
    gamma=0.5,                  
    
    scale_pos_weight=1,         

    random_state=42,
    n_jobs=-1,                  
    eval_metric='mlogloss',    
    early_stopping_rounds=20,  
    verbose=False
)

print("Training XGBoost model...")
print("Parameters:")
print(f"  - Trees: {xgb_model.n_estimators}")
print()

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("✅ XGBoost Model Training Complete")
print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Number of trees used: {len(xgb_model.get_booster().get_dump())}")

Training XGBoost model...
Parameters:
  - Trees: 200

✅ XGBoost Model Training Complete
Best iteration: 198
Number of trees used: 3200


In [ ]:
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f"Model Accuracy: {accuracy * 100:.2f}%")


Model Accuracy: 97.95%


In [64]:
labels_in_test = np.unique(y_test)

report = classification_report(
    y_test,
    y_pred,
    labels=labels_in_test,
    target_names=label_encoder.inverse_transform(labels_in_test)
)

print(report)

              precision    recall  f1-score   support

        أسمي       1.00      0.60      0.75        50
         أنا       1.00      1.00      1.00       100
        أهلا       1.00      1.00      1.00        25
       اتفضل       0.99      1.00      1.00       150
        الله       1.00      1.00      1.00        50
         بحب       1.00      1.00      1.00        75
      بركاته       1.00      1.00      1.00        25
        رحمة       0.90      1.00      0.95       150
        سلام       1.00      1.00      1.00        25
        عاوز       0.99      1.00      1.00       125
        مهذب       1.00      1.00      1.00        75
       نتعرف       1.00      1.00      1.00        50
         وعد       1.00      1.00      1.00        75

   micro avg       0.98      0.98      0.98       975
   macro avg       0.99      0.97      0.98       975
weighted avg       0.98      0.98      0.98       975



In [65]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Model Accuracy: 97.95%


In [ ]:
joblib.dump(xgb_model, 'TwoHands_XGBoost_Model.pkl')
joblib.dump(scaler, 'TwoHands_Scaler.pkl')
joblib.dump(label_encoder, 'TwoHands_LabelEncoder.pkl')

print("✅ Model saved successfully!")
print(f"Model file: TwoHands_XGBoost_Model.pkl")
print(f"Scaler file: TwoHands_Scaler.pkl")
print(f"Label encoder: TwoHands_LabelEncoder.pkl")

model_info = {
    'model_type': 'XGBoost',
    'num_features': len(feature_columns),
    'feature_names': feature_columns,
    'num_classes': len(label_encoder.classes_),
    'classes': label_encoder.classes_.tolist(),
    'accuracy': accuracy,
    'xgb_version': xgb.__version__
}

import json
with open('model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)

print("\n✅ Model metadata saved to model_metadata.json")

✅ Model saved successfully!
Model file: TwoHands_XGBoost_Model.pkl
Scaler file: TwoHands_Scaler.pkl
Label encoder: TwoHands_LabelEncoder.pkl

✅ Model metadata saved to model_metadata.json


In [ ]:
import os
import numpy as np
import joblib

if not os.path.exists('TwoHands_XGBoost_Model.pkl'):
    print("Model not found! Please run Cells 1-18 first")
else:
    loaded_model = joblib.load('TwoHands_XGBoost_Model.pkl')
    loaded_scaler = joblib.load('TwoHands_Scaler.pkl')
    loaded_encoder = joblib.load('TwoHands_LabelEncoder.pkl')
    
    print(f"✅ Two-hands model loaded successfully\n")
    
    def predict_gesture(sensor_values):
        """
        Predict gesture from 22 sensor values (two hands)
        
        Input order (22 values):
        Left Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        Right Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        """
        readings = np.array(sensor_values).reshape(1, -1)
        scaled = loaded_scaler.transform(readings)
        prediction = loaded_model.predict(scaled)[0]
        predicted_gesture = loaded_encoder.inverse_transform([prediction])[0]
        return predicted_gesture
    
    GlovesInput_numbers = [
        750, 780, 800, 820, 790,    
        -3000, -3200, -15200,        
        100, 150, -400,              
        
        780, 800, 820, 840, 810,    
        -1200, 1450, -14500,         
        -100, -80, 200               
    ]
    
    print("Input numbers (22 values - two hands):")
    print("[750, 780, 800, 820, 790, -3000, -3200, -15200, 100, 150, -400, 780, 800, 820, 840, 810, -1200, 1450, -14500, -100, -80, 200]")
    
    result = predict_gesture(GlovesInput_numbers)
    print(f"\nExpected: نتعرف")
    print(f"\nPredicted Gesture: {result}")

✅ Two-hands model loaded successfully

Input numbers (22 values - two hands):
[750, 780, 800, 820, 790, -3000, -3200, -15200, 100, 150, -400, 780, 800, 820, 840, 810, -1200, 1450, -14500, -100, -80, 200]

Expected: نتعرف

Predicted Gesture: نتعرف


In [ ]:
import os
import numpy as np
import joblib

if not os.path.exists('TwoHands_XGBoost_Model.pkl'):
    print("Model not found! Please run Cells 1-18 first")
else:
    loaded_model = joblib.load('TwoHands_XGBoost_Model.pkl')
    loaded_scaler = joblib.load('TwoHands_Scaler.pkl')
    loaded_encoder = joblib.load('TwoHands_LabelEncoder.pkl')
    
    print(f"✅ Two-hands model loaded successfully\n")
    
    def predict_gesture(sensor_values):
        """
        Predict gesture from 22 sensor values (two hands)
        
        Input order (22 values):
        Left Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        Right Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        """
        readings = np.array(sensor_values).reshape(1, -1)
        scaled = loaded_scaler.transform(readings)
        prediction = loaded_model.predict(scaled)[0]
        predicted_gesture = loaded_encoder.inverse_transform([prediction])[0]
        return predicted_gesture
    
    GlovesInput_numbers = [
        945, 910, 945, 790, 955,    
        -3500, -2000, -15500,        
        500, 200, -300,              
        
        810, 830, 650, 720, 820,    
        -8000, 12000, -2000,         
        -600, -4000, -6000              
    ]
    
    
    result = predict_gesture(GlovesInput_numbers)
    print(f"\nExpected: أنا")
    print(f"\nPredicted Gesture: {result}")

✅ Two-hands model loaded successfully


Expected: أنا

Predicted Gesture: أنا


In [78]:
import os
import numpy as np
import joblib

if not os.path.exists('TwoHands_XGBoost_Model.pkl'):
    print("Model not found! Please run Cells 1-18 first")
else:
    loaded_model = joblib.load('TwoHands_XGBoost_Model.pkl')
    loaded_scaler = joblib.load('TwoHands_Scaler.pkl')
    loaded_encoder = joblib.load('TwoHands_LabelEncoder.pkl')
    
    print(f"✅ Two-hands model loaded successfully\n")
    
    def predict_gesture(sensor_values):
        """
        Predict gesture from 22 sensor values (two hands)
        
        Input order (22 values):
        Left Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        Right Hand: flex1, flex2, flex3, flex4, flex5, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
        """
        readings = np.array(sensor_values).reshape(1, -1)
        scaled = loaded_scaler.transform(readings)
        prediction = loaded_model.predict(scaled)[0]
        predicted_gesture = loaded_encoder.inverse_transform([prediction])[0]
        return predicted_gesture
    
    GlovesInput_numbers = [
        870, 745, 665, 485, 565,    
        -3600, -2100, -15400,        
        490, 190, -310,              
        
        825, 448, 449, 585, 653,   
        -10700, 13050, 1250,         
        -440, -2250, -510            
    ]
    
    print("Input numbers (22 values - two hands) - APPROXIMATE values for 'سلام':")
    print("[870, 745, 665, 485, 565, -3600, -2100, -15400, 490, 190, -310, 825, 448, 449, 585, 653, -10700, 13050, 1250, -440, -2250, -510]")
    
    result = predict_gesture(GlovesInput_numbers)
    print(f"\nExpected: سلام")
    print(f"\nPredicted Gesture: {result}")

✅ Two-hands model loaded successfully

Input numbers (22 values - two hands) - APPROXIMATE values for 'سلام':
[870, 745, 665, 485, 565, -3600, -2100, -15400, 490, 190, -310, 825, 448, 449, 585, 653, -10700, 13050, 1250, -440, -2250, -510]

Expected: سلام

Predicted Gesture: سلام
